Загружаем датасет

In [10]:
import os
import psycopg
import dotenv
import pandas as pd

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
dotenv.load_dotenv()
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])

connection.update(postgres_credentials)
TABLE_NAME = "clean_flats_price"


with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]


df = pd.DataFrame(data, columns=columns)
df.head()

,id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.9,19.900000,1,False,False,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.0,16.600000,1,False,False,43.000000,13500000.0,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.0,32.000000,2,False,False,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.1,43.099998,3,False,False,76.000000,20000000.0,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.0,14.000000,1,False,False,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,True


In [11]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing  import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

X = df.drop(columns = ['price', 'building_id'])
y = df['price']

cat_features = df.select_dtypes(['object']).columns.tolist()
cat_features.append("building_type_int")
preprocessor = ColumnTransformer([
        ("binary", OneHotEncoder(drop="if_binary", sparse_output=False), cat_features)],
        remainder="passthrough",
        verbose_feature_names_out=False
    )
model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2,
                              max_features="sqrt", random_state=42, n_jobs=-1)
pipeline = Pipeline(
        [
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )



In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

train_size = 0.8
shuffle=True
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size, random_state=42, shuffle=shuffle)

pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
metrics = {
    "rmse": mean_squared_error(y_test, pred, squared=False),
    "mae": mean_absolute_error(y_test, pred),
    "r2": r2_score(y_test, pred)
}

print(metrics)


{'rmse': 2795542.821311236, 'mae': 2007594.9329998526, 'r2': 0.8019331564017734}


In [13]:
import mlflow


EXPERIMENT_NAME = "flats_price"
RUN_NAME = "base_model_registry"
REGISTRY_MODEL_NAME = "flats_price_baseline"

dotenv.load_dotenv()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, pred)
input_example = X_test[:10]
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_params(model.get_params())
    mlflow.log_param("seed", 42)
    mlflow.log_param("split_shuffle", True)
    mlflow.log_param("train_size", 0.8)
    mlflow.log_param("onehot_drop", "if_binary")

    mlflow.log_metrics(metrics)
    model_info = mlflow.sklearn.log_model(sk_model=pipeline, pip_requirements=pip_requirements,
                                          signature=signature, input_example=input_example,
                                          registered_model_name=REGISTRY_MODEL_NAME, artifact_path="models",
                                          await_registration_for=60)

/home/mle-user/mle_projects/mle-project-sprint-2-v001/venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'flats_price_baseline' already exists. Creating a new version of this model...
2026/08/16 19:41:45 INFO mlflow.tracking.